# 🔴 Solution: Causal Self-Attention（面试版）

## 📋 题目描述

**难度：** Medium

实现因果（掩码）自注意力。

与标准注意力类似，但阻止每个位置关注未来位置，这对 GPT 等自回归模型至关重要。

**签名:** `causal_attention(Q, K, V) -> Tensor`

**参数:**
- `Q` — 查询张量 (B, S, D)
- `K` — 键张量 (B, S, D)
- `V` — 值张量 (B, S, D)

**返回:** 因果掩码注意力输出 (B, S, D)

**约束:**
- 在 softmax 之前用 `-inf` 掩盖未来位置
- 位置 0 只能看到自身

**提示：** 标准注意力 + 因果遮蔽。`torch.triu(ones, diagonal=1)` 得到上三角 → softmax 前填 `-inf`。


In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def causal_attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor) -> torch.Tensor:
    d_k = key.size(-1)

    # ---- Step 1: 注意力分数 ----
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_k)

    # ---- Step 2: 因果 mask ----
    # torch.triu 上三角，diagonal=1 表示不包含对角线
    # 结果：位置 i 只能关注 j ≤ i
    seq_len = query.size(-2)
    causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=query.device), diagonal=1).bool()

    # ---- Step 3: mask + softmax ----
    # masked_fill(condition, value)：条件为 True 的位置填 value
    scores = scores.masked_fill(causal_mask, float('-inf'))

    # 手写 softmax
    scores_max = scores.max(dim=-1, keepdim=True).values
    exp_scores = torch.exp(scores - scores_max)
    attn_weights = exp_scores / exp_scores.sum(dim=-1, keepdim=True)

    return attn_weights @ value

# 面试追问：
# Q: 为什么需要 causal mask？
# A: 自回归生成时，预测第 t 个 token 只能用 0..t-1 的信息
# Q: triu 的 diagonal 参数？
# A: diagonal=0 包含对角线，diagonal=1 不包含

In [ ]:
# Verify
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))

## 📝 核心思路总结

1. **因果性**：每个位置只能看到自己和之前的位置
2. **上三角 mask**：torch.triu(diagonal=1) 生成未来位置的 mask
3. **-inf → softmax → 0**：mask 的标准实现方式
4. **自回归基础**：GPT 等生成模型的核心机制